# 19 — Robustesse à l'intensité de dégradation (à lancer SUR LE SERVEUR)

Charge les checkpoints `.pth` (baselines + 4 stratégies) et les évalue sur des jeux de test dégradés à intensités croissantes (bruit σ, sous-échantillonnage), pour les **3 datasets**.

⚠️ Nécessite les modèles `.pth` (présents sur le serveur, pas dans le snapshot local). Utilise `env_config` (chemins serveur). Sortie : `results/comparison/`.

## 1) Imports + modules partagés

In [1]:
import os, sys, json, glob, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import importlib
import env_config, degradation, cost
importlib.reload(env_config); importlib.reload(degradation); importlib.reload(cost)
from degradation import DegradedDataset, clean_tensor_transform, hf_transform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## 2) Configuration

In [2]:
DATASETS = ["Animals-10", "Imagewoof", "Intel"]

SIGMA_LEVELS = [0.0, 0.05, 0.10, 0.15, 0.25, 0.35]
SIGMA_FIXED_DOWNSCALE = 64
DOWNSCALE_LEVELS = [224, 128, 96, 64, 48, 32]
DOWNSCALE_FIXED_SIGMA = 0.15
JPEG_QUALITY = 60
BATCH_SIZE = 64
NUM_WORKERS = min(8, os.cpu_count() or 2)

# Modèles évalués (stem du .pth -> label). Décommenter les *_optimized pour les inclure
# (double le temps de calcul).
MODEL_LABELS = {
    "model_baseline_HF":                       "BL1 (HF)",
    "model_baseline_BF":                       "BL2 (BF)",
    "model_baseline_MIXTE":                    "BL3 (Mixte)",
    "model_strategy1_transfer_learning":       "S1 Transfer",
    "model_strategy2_cotraining_reweighting":  "S2 CoTrain",
    "model_strategy3_curriculum_learning":     "S3 Curriculum",
    "model_strategy5_ewc":                     "S5 EWC",
    # "model_strategy1_transfer_learning_optimized": "S1 +Optuna",
    # "model_strategy2_cotraining_reweighting_optimized": "S2 +Optuna",
    # "model_strategy3_curriculum_learning_optimized": "S3 +Optuna",
    # "model_strategy5_ewc_optimized": "S5 +Optuna",
}
COLORS = {"BL1 (HF)": "#9aa7b8", "BL2 (BF)": "#6e7a8a", "BL3 (Mixte)": "#3b4a5a",
          "S1 Transfer": "#4C72B0", "S2 CoTrain": "#55A868",
          "S3 Curriculum": "#C44E52", "S5 EWC": "#8172B2"}

OUT_DIR = env_config.comparison_dir()
os.makedirs(OUT_DIR, exist_ok=True)
print("Sortie :", OUT_DIR)

Sortie : /home/ivasic/PF22_MultiFidelity_DL/results/comparison


## 3) Utilitaires (chargement modèle robuste, évaluation dégradée)

In [3]:
def num_classes_of(test_dir):
    return len(datasets.ImageFolder(test_dir, transform=clean_tensor_transform()).classes)


def load_model(path, n_classes):
    """Charge un ResNet-18 ; gère fc=Linear ou fc=Sequential(Dropout, Linear)."""
    sd = torch.load(path, map_location=device)
    m = models.resnet18(weights=None)
    if any(k.startswith("fc.1.") for k in sd):
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, n_classes))
    else:
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    m.load_state_dict(sd)
    return m.to(device).eval()


@torch.no_grad()
def eval_degraded(model, test_dir, downscale, sigma):
    base = datasets.ImageFolder(test_dir, transform=clean_tensor_transform())
    ds = DegradedDataset(base, downscale=downscale, sigma=sigma,
                         jpeg_quality=JPEG_QUALITY, seeded=True)
    ld = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    correct = total = 0
    for x, y in ld:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total


@torch.no_grad()
def eval_clean(model, test_dir):
    base = datasets.ImageFolder(test_dir, transform=hf_transform())
    ld = DataLoader(base, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    correct = total = 0
    for x, y in ld:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total

## 4) Balayages (peut prendre plusieurs heures sur GPU)

In [4]:
results = {}
t0 = time.time()
for ds in DATASETS:
    try:
        base_dir = env_config.ensure_dataset_ready(ds)
    except Exception as e:
        print("SKIP", ds, ":", e); continue
    test_dir = os.path.join(base_dir, "test")
    res_dir = env_config.results_dir(ds)
    n = num_classes_of(test_dir)
    found = [os.path.basename(p) for p in glob.glob(os.path.join(res_dir, "model_*.pth"))]
    print(f"\n=== {ds} (num_classes={n}) | .pth dispo: {found} ===")

    dd = {"sigma_levels": SIGMA_LEVELS, "sigma_fixed_downscale": SIGMA_FIXED_DOWNSCALE,
          "downscale_levels": DOWNSCALE_LEVELS, "downscale_fixed_sigma": DOWNSCALE_FIXED_SIGMA,
          "jpeg_quality": JPEG_QUALITY, "models": {}}
    for stem, label in MODEL_LABELS.items():
        p = os.path.join(res_dir, stem + ".pth")
        if not os.path.exists(p):
            print(f"  [absent] {label}"); continue
        m = load_model(p, n)
        clean = eval_clean(m, test_dir)
        sig = [eval_degraded(m, test_dir, SIGMA_FIXED_DOWNSCALE, s) for s in SIGMA_LEVELS]
        dwn = [eval_degraded(m, test_dir, d, DOWNSCALE_FIXED_SIGMA) for d in DOWNSCALE_LEVELS]
        dd["models"][label] = {"color": COLORS.get(label, "#888"), "clean_HF_acc": clean,
                               "sigma_acc": sig, "downscale_acc": dwn}
        print(f"  {label:14s} cleanHF={clean:5.1f} | sigma {sig[0]:.1f}->{sig[-1]:.1f} | down {dwn[0]:.1f}->{dwn[-1]:.1f}")
    results[ds] = dd

with open(os.path.join(OUT_DIR, "robustness_degradation.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nTerminé en {(time.time()-t0)/60:.1f} min -> {os.path.join(OUT_DIR, 'robustness_degradation.json')}")


=== Animals-10 (num_classes=10) | .pth dispo: ['model_strategy5_ewc.pth', 'model_strategy2_cotraining_reweighting.pth', 'model_baseline_MIXTE.pth', 'model_strategy5_ewc_optimized.pth', 'model_strategy1_transfer_learning_optimized.pth', 'model_baseline_HF.pth', 'model_strategy3_student.pth', 'model_strategy3_curriculum_learning.pth', 'model_baseline_BF.pth', 'model_strategy1_transfer_learning.pth', 'model_strategy3_curriculum_learning_optimized.pth', 'model_strategy3_teacher.pth', 'model_strategy2_cotraining_reweighting_optimized.pth'] ===


/tmp/ipykernel_3292364/3926663295.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(path, map_location=device)


  BL1 (HF)       cleanHF= 53.0 | sigma 28.7->15.0 | down 35.0->20.8


  BL2 (BF)       cleanHF= 54.3 | sigma 71.3->34.2 | down 48.7->59.2


  BL3 (Mixte)    cleanHF= 70.5 | sigma 69.1->34.6 | down 65.1->59.9


...

  BL3 (Mixte)    cleanHF= 83.4 | sigma 82.2->67.9 | down 82.1->77.1


  S1 Transfer    cleanHF= 86.3 | sigma 80.5->57.9 | down 84.0->61.4


  S2 CoTrain     cleanHF= 80.9 | sigma 76.2->65.4 | down 78.3->71.0


  S3 Curriculum  cleanHF= 84.3 | sigma 81.5->70.1 | down 83.1->75.2


  S5 EWC         cleanHF= 86.2 | sigma 82.8->66.6 | down 84.8->66.1

Terminé en 16.7 min -> /home/ivasic/PF22_MultiFidelity_DL/results/comparison/robustness_degradation.json


## 5) Figures (une par dataset)

In [5]:
for ds, r in results.items():
    if not r["models"]:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    fig.suptitle(f"Robustesse à la dégradation — {ds}", fontsize=14)
    ax = axes[0]
    for lbl, m in r["models"].items():
        ax.plot(r["sigma_levels"], m["sigma_acc"], marker="o", color=m["color"], label=lbl)
    ax.set_title(f"Bruit gaussien (downsample={r['sigma_fixed_downscale']}px, JPEG q{r['jpeg_quality']})")
    ax.set_xlabel("σ du bruit (→ plus dégradé)"); ax.set_ylabel("Précision test (%)")
    ax.set_ylim(0, 100); ax.grid(alpha=0.3); ax.legend(fontsize=7)
    ax = axes[1]
    for lbl, m in r["models"].items():
        ax.plot(r["downscale_levels"], m["downscale_acc"], marker="s", color=m["color"], label=lbl)
    ax.set_title(f"Sous-échantillonnage (σ={r['downscale_fixed_sigma']}, JPEG q{r['jpeg_quality']})")
    ax.set_xlabel("Résolution intermédiaire (px) — gauche = plus dégradé")
    ax.set_ylabel("Précision test (%)"); ax.set_ylim(0, 100); ax.grid(alpha=0.3); ax.legend(fontsize=7)
    suffix = ds.lower().replace("-", "")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    out = os.path.join(OUT_DIR, f"robustness_degradation_{suffix}.png")
    plt.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    print("Figure:", out)

Figure: /home/ivasic/PF22_MultiFidelity_DL/results/comparison/robustness_degradation_animals10.png


Figure: /home/ivasic/PF22_MultiFidelity_DL/results/comparison/robustness_degradation_imagewoof.png


Figure: /home/ivasic/PF22_MultiFidelity_DL/results/comparison/robustness_degradation_intel.png


## Note

Une courbe qui décroît lentement = modèle robuste à la dégradation ; une chute brutale = modèle fragile (sur-spécialisé sur le domaine propre). Une fois terminé, rapatrie `results/comparison/robustness_degradation*.png` et le JSON pour le rapport.